# audEERING emotion scores for ProMoNet outputs

This notebook runs `audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim` over the canonical natural recordings and the generated ProMoNet and Praat WAV files. It saves one tidy CSV for later acoustic and emotion-distance analysis.

The model produces scores in this exact order: **arousal, dominance, valence**. It is intended for non-commercial research and is distributed under CC BY-NC-SA 4.0. The first run downloads the pinned model revision to the normal Hugging Face cache; later runs reuse that cache.

In [ ]:
from __future__ import annotations

import os
import re
import warnings
from datetime import datetime, timezone
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import transformers
from IPython.display import display
from tqdm.auto import tqdm
from transformers import Wav2Vec2Processor
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2Model,
    Wav2Vec2PreTrainedModel,
)

MODEL_ID = "audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim"
MODEL_REVISION = "6eba34a2485ea31cb03600241787c3a5edab8626"
INFERENCE_SAMPLE_RATE = 16_000

# These defaults can be overridden before running the notebook, or with
# AUDEERING_DEVICE, AUDEERING_FORCE_RECOMPUTE, AUDEERING_MAX_FILES, and
# AUDEERING_CHECKPOINT_EVERY environment variables.
DEVICE = os.environ.get("AUDEERING_DEVICE", "auto")
FORCE_RECOMPUTE = os.environ.get("AUDEERING_FORCE_RECOMPUTE", "false").lower() in {
    "1", "true", "yes"
}
_max_files = os.environ.get("AUDEERING_MAX_FILES")
MAX_FILES = int(_max_files) if _max_files else None
CHECKPOINT_EVERY = int(os.environ.get("AUDEERING_CHECKPOINT_EVERY", "10"))
if MAX_FILES is not None and MAX_FILES <= 0:
    raise ValueError("MAX_FILES must be a positive integer or None")
if CHECKPOINT_EVERY <= 0:
    raise ValueError("CHECKPOINT_EVERY must be positive")

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "audio").is_dir() and (candidate / "outputs").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not find the ProMoNet root above {start}. "
        "Start Jupyter from the project root or its analysis folder."
    )

PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "audEERING"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_FILENAME = (
    "audeering_scores.csv"
    if MAX_FILES is None
    else f"audeering_scores_smoke_{MAX_FILES}.csv"
)
RESULTS_PATH = OUTPUT_DIR / RESULTS_FILENAME

print(f"Project root: {PROJECT_ROOT}")
print(f"Results path: {RESULTS_PATH}")
print(f"torch={torch.__version__}, transformers={transformers.__version__}")
print(
    f"DEVICE={DEVICE!r}, FORCE_RECOMPUTE={FORCE_RECOMPUTE}, "
    f"MAX_FILES={MAX_FILES}, CHECKPOINT_EVERY={CHECKPOINT_EVERY}"
)

## 1. Discover and classify audio

Natural data are limited to the 50 matched happy, neutral, and sad sentences. Generated audio is read only from the two top-level output pipelines. Sentence 39's happy edit is an intentional exclusion because its aligned neutral and happy vowels do not match.

In [ ]:
NATURAL_PATTERN = re.compile(
    r"^(?P<speaker>\d+)_(?P<sentence>\d{2})_(?P<emotion>hap|neu|sad)_[fm]$"
)
PROMONET_EDIT_PATTERN = re.compile(
    r"^(?P<speaker>\d+)_(?P<sentence>\d{2})_neu_[fm]__from__"
    r"(?P=speaker)_(?P=sentence)_(?P<emotion>hap|sad)_[fm]__"
)
PROMONET_RECON_PATTERN = re.compile(
    r"^(?P<speaker>\d+)_(?P<sentence>\d{2})_neu_[fm]__reconstruction__"
)
PRAAT_EDIT_PATTERN = re.compile(
    r"^(?P<speaker>\d+)_(?P<sentence>\d{2})_neu_from_(?P<emotion>hap|sad)__"
)

EXPECTED_GROUP_COUNTS = {
    ("natural", "natural", "hap"): 50,
    ("natural", "natural", "neu"): 50,
    ("natural", "natural", "sad"): 50,
    ("promonet", "edit", "hap"): 49,
    ("promonet", "edit", "sad"): 50,
    ("promonet", "reconstruction", "neu"): 50,
    ("praat", "edit", "hap"): 49,
    ("praat", "edit", "sad"): 50,
}

def file_metadata(path: Path) -> dict[str, object]:
    stat = path.stat()
    return {
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "filename": path.name,
        "file_size_bytes": int(stat.st_size),
        "modified_time_utc": datetime.fromtimestamp(
            stat.st_mtime, tz=timezone.utc
        ).isoformat(),
        "_path": path,
    }

def parse_natural(path: Path) -> dict[str, object] | None:
    match = NATURAL_PATTERN.fullmatch(path.stem)
    if match is None:
        return None
    emotion = match.group("emotion")
    return {
        **file_metadata(path),
        "system": "natural",
        "condition": "natural",
        "speaker_id": match.group("speaker"),
        "sentence_id": match.group("sentence"),
        "source_emotion": emotion,
        "target_emotion": emotion,
    }

def parse_promonet(path: Path) -> dict[str, object]:
    edit = PROMONET_EDIT_PATTERN.match(path.stem)
    if edit is not None:
        return {
            **file_metadata(path),
            "system": "promonet",
            "condition": "edit",
            "speaker_id": edit.group("speaker"),
            "sentence_id": edit.group("sentence"),
            "source_emotion": "neu",
            "target_emotion": edit.group("emotion"),
        }
    reconstruction = PROMONET_RECON_PATTERN.match(path.stem)
    if reconstruction is not None:
        return {
            **file_metadata(path),
            "system": "promonet",
            "condition": "reconstruction",
            "speaker_id": reconstruction.group("speaker"),
            "sentence_id": reconstruction.group("sentence"),
            "source_emotion": "neu",
            "target_emotion": "neu",
        }
    raise ValueError(f"Unrecognized ProMoNet output filename: {path.name}")

def parse_praat(path: Path) -> dict[str, object]:
    match = PRAAT_EDIT_PATTERN.match(path.stem)
    if match is None:
        raise ValueError(f"Unrecognized Praat output filename: {path.name}")
    return {
        **file_metadata(path),
        "system": "praat",
        "condition": "edit",
        "speaker_id": match.group("speaker"),
        "sentence_id": match.group("sentence"),
        "source_emotion": "neu",
        "target_emotion": match.group("emotion"),
    }

def discover_audio() -> pd.DataFrame:
    rows: list[dict[str, object]] = []
    for path in sorted((PROJECT_ROOT / "audio").glob("*.wav")):
        parsed = parse_natural(path)
        if parsed is not None:
            rows.append(parsed)
    for path in sorted((PROJECT_ROOT / "outputs" / "vowel_edit_pipeline").rglob("*.wav")):
        rows.append(parse_promonet(path))
    for path in sorted((PROJECT_ROOT / "outputs" / "praat_pipeline").rglob("*.wav")):
        rows.append(parse_praat(path))

    inventory = pd.DataFrame(rows)
    if inventory.empty:
        raise FileNotFoundError("No matching WAV files were discovered")
    duplicates = inventory["relative_path"].duplicated(keep=False)
    if duplicates.any():
        raise ValueError(
            "Duplicate audio paths discovered:\n"
            + inventory.loc[duplicates, "relative_path"].to_string(index=False)
        )
    return inventory.sort_values(
        ["system", "condition", "target_emotion", "sentence_id", "relative_path"]
    ).reset_index(drop=True)

inventory = discover_audio()
inventory_counts = inventory.groupby(
    ["system", "condition", "target_emotion"], dropna=False
).size().rename("wav_count")
display(inventory_counts.to_frame())

actual_counts = inventory_counts.to_dict()
if actual_counts != EXPECTED_GROUP_COUNTS:
    warnings.warn(
        "Discovered groups differ from the current expected 398-file dataset. "
        f"Expected {EXPECTED_GROUP_COUNTS}; found {actual_counts}. "
        "Known additional files will still be scored."
    )
else:
    print("Inventory matches the expected 398 files.")

## 2. Load the pinned audEERING model

The custom model class below follows audEERING's published Hugging Face example. Model loading is lazy: a fully cached rerun does not load the neural network.

In [ ]:
class RegressionHead(nn.Module):
    """Regression head published with the audEERING model card."""

    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.dropout = nn.Dropout(config.final_dropout)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, features, **kwargs):
        features = self.dropout(features)
        features = self.dense(features)
        features = torch.tanh(features)
        features = self.dropout(features)
        return self.out_proj(features)

class EmotionModel(Wav2Vec2PreTrainedModel):
    """Wav2Vec2 encoder with audEERING's dimensional regression head."""

    def __init__(self, config):
        super().__init__(config)
        self.config = config
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = RegressionHead(config)
        # Transformers 5.x finalizes tied-weight bookkeeping in post_init().
        self.post_init()

    def forward(self, input_values):
        hidden_states = self.wav2vec2(input_values)[0]
        pooled_states = torch.mean(hidden_states, dim=1)
        logits = self.classifier(pooled_states)
        return pooled_states, logits

def resolve_device(requested: str) -> torch.device:
    requested = requested.lower()
    if requested == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if requested.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but torch.cuda.is_available() is False")
    return torch.device(requested)

def load_emotion_model(device: torch.device):
    print(f"Loading {MODEL_ID}@{MODEL_REVISION} on {device} ...")
    processor = Wav2Vec2Processor.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION
    )
    model = EmotionModel.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION
    ).to(device=device, dtype=torch.float32)
    model.eval()
    return processor, model

ACTIVE_DEVICE = resolve_device(DEVICE)
print(f"Resolved inference device: {ACTIVE_DEVICE}")

## 3. Score audio with resumable checkpoints

Successful unchanged rows are reused. Error rows, new files, changed files, and files produced with another model revision are processed again. A limited smoke test writes to a separate CSV and therefore cannot overwrite the full table.

In [ ]:
RESULT_COLUMNS = [
    "relative_path",
    "filename",
    "system",
    "condition",
    "speaker_id",
    "sentence_id",
    "source_emotion",
    "target_emotion",
    "original_sample_rate",
    "original_channels",
    "duration_seconds",
    "file_size_bytes",
    "modified_time_utc",
    "model_id",
    "model_revision",
    "inference_sample_rate",
    "device",
    "processed_at_utc",
    "arousal",
    "dominance",
    "valence",
    "status",
    "error_message",
]
IDENTITY_COLUMNS = [
    "relative_path",
    "filename",
    "system",
    "condition",
    "speaker_id",
    "sentence_id",
    "source_emotion",
    "target_emotion",
    "file_size_bytes",
    "modified_time_utc",
]

def cache_key(row: dict[str, object] | pd.Series) -> tuple[object, ...]:
    return (
        str(row["relative_path"]),
        int(row["file_size_bytes"]),
        str(row["modified_time_utc"]),
        str(row.get("model_revision", MODEL_REVISION)),
    )

def load_existing_results() -> pd.DataFrame:
    if FORCE_RECOMPUTE or not RESULTS_PATH.is_file():
        return pd.DataFrame(columns=RESULT_COLUMNS)
    existing = pd.read_csv(
        RESULTS_PATH,
        dtype={"speaker_id": "string", "sentence_id": "string"},
    )
    missing = sorted(set(RESULT_COLUMNS) - set(existing.columns))
    if missing:
        warnings.warn(
            f"Ignoring incompatible existing results; missing columns: {missing}"
        )
        return pd.DataFrame(columns=RESULT_COLUMNS)
    return existing[RESULT_COLUMNS].copy()

def ordered_results(rows: list[dict[str, object]]) -> pd.DataFrame:
    frame = pd.DataFrame(rows)
    for column in RESULT_COLUMNS:
        if column not in frame:
            frame[column] = pd.NA
    if frame.empty:
        return frame[RESULT_COLUMNS]
    frame = frame[RESULT_COLUMNS]
    return frame.sort_values(
        ["system", "condition", "target_emotion", "sentence_id", "relative_path"]
    ).reset_index(drop=True)

def write_results(rows: list[dict[str, object]]) -> pd.DataFrame:
    frame = ordered_results(rows)
    temporary_path = RESULTS_PATH.with_name(RESULTS_PATH.name + ".tmp")
    frame.to_csv(temporary_path, index=False)
    os.replace(temporary_path, RESULTS_PATH)
    return frame

def score_audio(
    row: pd.Series,
    processor: Wav2Vec2Processor,
    model: EmotionModel,
    device: torch.device,
) -> dict[str, object]:
    path = Path(row["_path"])
    info = sf.info(path)
    signal, loaded_sample_rate = librosa.load(
        path, sr=INFERENCE_SAMPLE_RATE, mono=True, dtype=np.float32
    )
    if loaded_sample_rate != INFERENCE_SAMPLE_RATE:
        raise ValueError(f"Unexpected librosa sample rate: {loaded_sample_rate}")
    if signal.size == 0 or not np.isfinite(signal).all():
        raise ValueError("Loaded audio is empty or contains non-finite samples")

    inputs = processor(
        signal, sampling_rate=INFERENCE_SAMPLE_RATE, return_tensors="pt"
    )["input_values"].to(device=device, dtype=torch.float32)
    with torch.inference_mode():
        _, logits = model(inputs)
    scores = logits.squeeze(0).detach().cpu().numpy().astype(float)
    if scores.shape != (3,) or not np.isfinite(scores).all():
        raise ValueError(f"Expected three finite scores; received shape {scores.shape}")

    result = {column: row[column] for column in IDENTITY_COLUMNS}
    result.update(
        {
            "original_sample_rate": int(info.samplerate),
            "original_channels": int(info.channels),
            "duration_seconds": float(info.duration),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
            "inference_sample_rate": INFERENCE_SAMPLE_RATE,
            "device": str(device),
            "processed_at_utc": datetime.now(timezone.utc).isoformat(),
            "arousal": float(scores[0]),
            "dominance": float(scores[1]),
            "valence": float(scores[2]),
            "status": "success",
            "error_message": "",
        }
    )
    return result

selected_inventory = (
    inventory if MAX_FILES is None else inventory.head(MAX_FILES).copy()
)
existing = load_existing_results()
existing_success = existing[
    (existing["status"] == "success")
    & (existing["model_id"] == MODEL_ID)
    & (existing["model_revision"] == MODEL_REVISION)
]
existing_by_key = {
    cache_key(row): row.to_dict() for _, row in existing_success.iterrows()
}

reused_rows: list[dict[str, object]] = []
pending_indices: list[int] = []
for index, row in selected_inventory.iterrows():
    key = (
        str(row["relative_path"]),
        int(row["file_size_bytes"]),
        str(row["modified_time_utc"]),
        MODEL_REVISION,
    )
    if not FORCE_RECOMPUTE and key in existing_by_key:
        reused_rows.append(existing_by_key[key])
    else:
        pending_indices.append(index)

print(f"Selected files: {len(selected_inventory)}")
print(f"Reused successful rows: {len(reused_rows)}")
print(f"Files requiring inference: {len(pending_indices)}")

new_rows: list[dict[str, object]] = []
processor = model = None
if pending_indices:
    processor, model = load_emotion_model(ACTIVE_DEVICE)

for completed, index in enumerate(
    tqdm(pending_indices, desc="audEERING inference"), start=1
):
    row = selected_inventory.loc[index]
    try:
        result = score_audio(row, processor, model, ACTIVE_DEVICE)
    except Exception as error:
        result = {column: row[column] for column in IDENTITY_COLUMNS}
        result.update(
            {
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
                "inference_sample_rate": INFERENCE_SAMPLE_RATE,
                "device": str(ACTIVE_DEVICE),
                "processed_at_utc": datetime.now(timezone.utc).isoformat(),
                "status": "error",
                "error_message": f"{type(error).__name__}: {error}",
            }
        )
        print(f"ERROR {row['relative_path']}: {result['error_message']}")
    new_rows.append(result)
    if completed % CHECKPOINT_EVERY == 0:
        write_results(reused_rows + new_rows)

if not pending_indices and RESULTS_PATH.is_file():
    results = ordered_results(reused_rows)
    print(f"All {len(results)} selected rows were reused; existing CSV left unchanged.")
else:
    results = write_results(reused_rows + new_rows)
    print(f"Saved {len(results)} rows to {RESULTS_PATH}")

## 4. Validate and preview saved scores

These are descriptive checks only. Target-distance calculations, confidence intervals, and plots belong in the later analysis notebook.

In [ ]:
if results["relative_path"].duplicated().any():
    raise AssertionError("The saved table contains duplicate relative paths")

status_counts = results["status"].value_counts(dropna=False).rename("row_count")
display(status_counts.to_frame())

successful = results[results["status"] == "success"].copy()
failed = results[results["status"] != "success"].copy()
if not failed.empty:
    warnings.warn(f"{len(failed)} files failed; rerun to retry them.")
    display(failed[["relative_path", "error_message"]])

score_columns = ["arousal", "dominance", "valence"]
if not successful.empty:
    if not np.isfinite(successful[score_columns].to_numpy(dtype=float)).all():
        raise AssertionError("Successful rows contain non-finite scores")
    outside = (
        (successful[score_columns] < 0.0) | (successful[score_columns] > 1.0)
    ).any(axis=1)
    if outside.any():
        warnings.warn(
            f"{int(outside.sum())} files have scores outside the model's "
            "approximate 0-1 range; values were preserved without clipping."
        )

    summary = successful.groupby(
        ["system", "condition", "target_emotion"]
    )[score_columns].agg(["count", "mean", "std", "min", "max"])
    display(summary)
    display(successful.head(10))

if MAX_FILES is None and len(inventory) == 398:
    if len(results) != 398 or len(successful) != 398:
        warnings.warn(
            f"Expected 398 successful full-run rows; found "
            f"{len(results)} total and {len(successful)} successful."
        )
    else:
        print("Full-run acceptance check passed: 398 unique successful rows.")